# CHLA: Covariate Binarization

This is the code used to binarize the CHLA dataset for the ESI project.
It uses the preprocessed data file 'preprocessed_data_with_triage_vitals.csv'.

Blanca Romero, 09/10/25

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# os.chdir('/Volumes/chip-lacava/Private/ch269952/CHLA') % CHANGE
os.getcwd()

'/Volumes/chip-lacava/Private/ch269952/CHLA'

In [2]:
# Load preprocessed data (not binarized)
data = pd.read_csv('preprocessed_data_with_triage_vitals.csv')

In [5]:
# Columns in preprocessed data
print(data.columns.tolist())

['id_visit', 'id_patient', 'disposition', 'age', 'age_group', 'sex', 'race_ethnicity', 'zip', 'state', 'miles_travelled', 'SDI_score', 'triage_acuity', 'raw_complaint', 'insurance', 'disposition_datetime', 'arrival_datetime', 'triage_datetime', 'minutes_since_first_arrival', 'arrival_year', 'arrival_season', 'arrival_day_type', 'arrival_time_block', 'arrival_mode', 'preferred_language', 'weight', 'num_labs', 'any_labs', 'num_meds', 'any_meds', 'num_IV_meds', 'any_IV_meds', 'triage_vitals_time.x', 'icd_score', 'crowdedness', 'num_previous_admissions', 'num_previous_visits_without_admission', 'triage_vitals_time.y', 'triage_heart_rate', 'triage_respiratory_rate', 'triage_oxygen_saturation', 'triage_temperature', 'triage_systolic_bp', 'triage_diastolic_bp', 'diagnosis_anemia', 'diagnosis_pain_conditions', 'diagnosis_congenital_malformations', 'diagnosis_cardiovascular', 'diagnosis_nausea_and_vomiting', 'diagnosis_epilepsy', 'diagnosis_developmental_delays', 'diagnosis_gastrointestinal', '

In [6]:
# Create empty data frame for binarized data
data_bin = pd.DataFrame(index=data.index)

# Visit ID
data_bin['id_visit'] = data['id_visit']
data_bin['id_patient'] = data['id_patient']
data_bin['age'] = data['age'] # keep raw age values for later use

In [7]:
# AGE (reference group < 3 months)
age_group_dummies = (pd.get_dummies(data['age_group'], prefix='is')).astype(int)
age_group_dummies = (age_group_dummies.rename(columns={'age_group_unknown': 'is_unknown_age'})).astype(int)
age_group_dummies = age_group_dummies.drop('is_older_than_15_years', axis=1)
data_bin = pd.concat([data_bin, age_group_dummies], axis=1)

In [8]:
# SEX (reference group: female)
data_bin['is_male'] = (data['sex'] == 'M').astype(int)

In [9]:
# RACE (reference group: non-hispanic white)
data_bin['is_asian'] = (data['race_ethnicity'] == 'asian').astype(int)
data_bin['is_hispanic'] = (data['race_ethnicity'] == 'hispanic').astype(int)
data_bin['is_hispanic_white'] = (data['race_ethnicity'] == 'hispanic_white').astype(int)
data_bin['is_non_hispanic_black'] = (data['race_ethnicity'] == 'non_hispanic_black').astype(int)
data_bin['is_other'] = (data['race_ethnicity'] == 'other').astype(int)
data_bin['is_unknown'] = (data['race_ethnicity'] == 'unknown').astype(int)

In [10]:
# MEANS OF ARRIVAL (reference group: self)
data_bin['arrival_EMS'] = (data['arrival_mode'] == 'EMS').astype(int)
data_bin['arrival_unknown'] = (data['arrival_mode'] == 'unknowm').astype(int)
data_bin['arrival_other'] = (data['arrival_mode'] == 'other').astype(int)

In [11]:
# PREFERRED LANGUAGE (reference group: English)
data_bin['is_language_spanish'] = (data['preferred_language'] == 'Spanish').astype(int)
data_bin['is_language_mandarin'] = (data['preferred_language'] == 'Mandarin').astype(int)
data_bin['is_language_russian'] = (data['preferred_language'] == 'Russian').astype(int)
data_bin['is_language_armenian'] = (data['preferred_language'] == 'Armenian').astype(int)
data_bin['is_language_other'] = (data['preferred_language'] == 'other_unknown').astype(int)

In [12]:
# GEOGRAPHIC DATA
data_bin['state_out_of_state'] = (data['state'] != 'CA').astype(int)
data_bin['state_unknown'] = data['state'].isna().astype(int)

data_bin['miles_travelled'] = data['miles_travelled']
mean_value = pd.to_numeric(data_bin['miles_travelled'], errors='coerce').mean() # replace NaN by mean value
data_bin['miles_travelled'] = data_bin['miles_travelled'].fillna(mean_value)

data_bin['SDI_score'] = data['SDI_score'].replace('unknown', np.nan) # replace 'NaN' by mean value
mean_value = pd.to_numeric(data_bin['SDI_score'], errors='coerce').mean() 
data_bin['SDI_score'] = data_bin['SDI_score'].fillna(mean_value)

In [13]:
# INSURANCE (reference group: Public)
data_bin['insurance_private'] = (data['insurance'] == 'Private').astype(int)
data_bin['insurance_unknown'] = (data['insurance'] == 'unknown').astype(int)

In [14]:
# ADMISSIONS/DISPOSITION (reference group:discharge)
data_bin['is_admitted'] = (data['disposition'] == 'Admitted').astype(int)
data_bin['is_transfer'] = (data['disposition'] == 'Transfer').astype(int)

In [15]:
# PRIOR VISITS (continuous variable)
prior_visits_col = ['num_previous_admissions', 'num_previous_visits_without_admission']
data_bin[prior_visits_col] = data[prior_visits_col]

In [16]:
# CROWDEDNESS (continuous variable)
data_bin['num_patients_at_arrival'] = data['crowdedness']

In [17]:
# COMORBIDITY SCORE (continuous variable)
data_bin['icd_score'] = data['icd_score']
mean_value = pd.to_numeric(data_bin['icd_score'], errors='coerce').mean() # replace NaN by mean value
data_bin['icd_score'] = data_bin['icd_score'].fillna(mean_value)

In [18]:
# WEIGHT (continuous variable, normalized weight by age group)
data_bin['weight'] = data['weight'].replace('not_taken', np.nan) # replace 'not_taken' by mean value
mean_value = pd.to_numeric(data_bin['weight'], errors='coerce').mean() 
data_bin['weight'] = data_bin['weight'].fillna(mean_value)

In [19]:
# TRIAGE VITALS

def get_vitals_rate_std_by_age(data, triage_vital_col):
    
    data = data.copy()
    
    # Calculate mean and std for each age group
    stats = data.groupby('age_group')[triage_vital_col].agg(['mean', 'std']).reset_index()
    
    # Merge back to original data
    data = data.merge(stats, on='age_group', suffixes=('', '_group'))
    
    # Calculate z-scores
    normalized_col = np.abs((data[triage_vital_col] - data['mean']) / data['std'])
    
    # Handle missing values
    normalized_col = np.where(data[triage_vital_col].isna(), 0, normalized_col)
    
    return normalized_col
    
# HEART RATE: std per age group
data_bin['norm_heart_rate'] = get_vitals_rate_std_by_age(data, 'triage_heart_rate')

# RESPIRATORY RATE: std per age group
data_bin['norm_respiratory_rate'] = get_vitals_rate_std_by_age(data, 'triage_respiratory_rate')

# Blood Pressure: score, distance from PALS
def get_pals_threshold(age_group, age):
    
    if age_group == 'under_3_months':
        return 60
    elif age_group in ['three_to_6_months', 'six_to_12_months']:
        return 70
    elif age_group in ['twelve_to_18_months', 'eighteen_months_to_3_years', 'three_to_5_years','five_to_10_years' ]:
        return 70 + age*2
    elif age_group in ['ten_to_15_years', 'older_than_15_years']:
        return 90

def calculate_systolic_bp_diff_pals(data):
    
    data = data.copy()
    systolic_bp = data['triage_systolic_bp']
    age_group = data['age_group']
    age = data['age']
    
    # Calculate PALS threshold 
    pals_threshold = data.apply(lambda row: get_pals_threshold(row['age_group'], row['age']), axis=1)
    
    # Calculate difference (systolic pressure above PALS = 0)
    data['syst_diff_pals'] = np.maximum(pals_threshold - systolic_bp, 0)
    
    # Missing values = mean of age group
    group_means = data.groupby('age_group')['syst_diff_pals'].mean()
    data['syst_diff_pals'] = data['syst_diff_pals'].fillna(data['age_group'].map(group_means))
    
    return data['syst_diff_pals']

data_bin['syst_diff_pals'] = calculate_systolic_bp_diff_pals(data)
mean_value = pd.to_numeric(data_bin['syst_diff_pals'], errors='coerce').mean() # replace NaN by mean value
data_bin['syst_diff_pals'] = data_bin['syst_diff_pals'].fillna(mean_value)

# TEMPERATURE: positive distance from 38C
fever_thresh = 38 # Celsius
data_bin['temp_fever'] = np.maximum(data['triage_temperature'] - fever_thresh, 0)
mean_value = pd.to_numeric(data_bin['temp_fever'], errors='coerce').mean() # replace NaN by mean value
data_bin['temp_fever'] = data_bin['temp_fever'].fillna(mean_value)

In [20]:
# TIME STAMPS
# Arrival year (reference group (2018))
data_bin['arrival_year_2019'] = (data['arrival_year'] == '2019').astype(int)
data_bin['arrival_year_2020'] = (data['arrival_year'] == '2020').astype(int)
data_bin['arrival_year_2021'] = (data['arrival_year'] == '2021').astype(int)
data_bin['arrival_year_2022'] = (data['arrival_year'] == '2022').astype(int)
data_bin['arrival_year_2023'] = (data['arrival_year'] == '2023').astype(int)
data_bin['arrival_year_2024'] = (data['arrival_year'] == '2024').astype(int)
data_bin['arrival_year_2025'] = (data['arrival_year'] == '2025').astype(int)

# Arrival season (reference group: Summer)
data_bin['arrival_season_winter'] = (data['arrival_season'] == 'Winter (Dec-Feb)').astype(int)
data_bin['arrival_season_fall']   = (data['arrival_season'] == 'Fall (Sep-Nov)').astype(int)
data_bin['arrival_season_spring'] = (data['arrival_season'] == 'Spring (Mar-May)').astype(int)

# Arrival day type (reference gorup: Weekday)
data_bin['arrival_weekend'] = (data['arrival_day_type'] == 'Weekend').astype(int)

# Arrival time block (reference group: 18:00-23:59)
data_bin['arrival_time_afternoon'] = (data['arrival_time_block'] == '12:00-17:59').astype(int)
data_bin['arrival_time_small_hours'] = (data['arrival_time_block'] == '00:00-05:59').astype(int)
data_bin['arrival_time_morning'] = (data['arrival_time_block'] == '06:00-11:59').astype(int)

In [21]:
# LABS & MEDS
labs_meds_col = ['num_labs', 'any_labs', 'num_meds', 'any_meds', 'num_IV_meds', 'any_IV_meds']
data_bin[labs_meds_col] = data[labs_meds_col]

In [22]:
# ADD OTHER RAW INFORMATION
columns_to_add = [
    'triage_acuity', 'triage_temperature', 'triage_systolic_bp', 
    'triage_diastolic_bp', 'raw_complaint', 'raw_reason_for_visit'
]
data_bin[columns_to_add] = data[columns_to_add]
data_bin['triage_hr'] = data['triage_heart_rate']
data_bin['triage_rr'] = data['triage_respiratory_rate']
data_bin['triage_spo2'] = data['triage_oxygen_saturation']

diagnosis_list = ['diagnosis_anemia', 'diagnosis_pain_conditions', 'diagnosis_congenital_malformations', 
                  'diagnosis_cardiovascular', 'diagnosis_nausea_and_vomiting', 'diagnosis_epilepsy', 
                  'diagnosis_developmental_delays', 'diagnosis_gastrointestinal', 'diagnosis_asthma', 
                  'diagnosis_sleep_disorders', 'diagnosis_anxiety', 'diagnosis_diabetes_mellitus', 
                  'diagnosis_joint_disorders', 'diagnosis_any_malignancy', 'diagnosis_chromosomal_anomalies', 
                  'diagnosis_weight_loss', 'diagnosis_eating_disorders', 'diagnosis_menstrual_disorders', 
                  'diagnosis_alcohol_abuse', 'diagnosis_depression', 'diagnosis_psychotic_disorders', 
                  'diagnosis_drug_abuse', 'diagnosis_conduct_disorders', 'diagnosis_smoking']
data_bin.loc[:, diagnosis_list] = data[diagnosis_list].fillna(0)

complaint_groups = ['complaint_contains_abdominal_pain', 'complaint_contains_assault', 
                    'complaint_contains_allergic_reaction', 'complaint_contains_altered_mental_status', 
                    'complaint_contains_asthma_or_wheezing', 'complaint_contains_bites_or_stings', 
                    'complaint_contains_burn', 'complaint_contains_cardiac', 'complaint_contains_chest_pain', 
                    'complaint_contains_congestion', 'complaint_contains_constipation', 
                    'complaint_contains_cough', 'complaint_contains_crying_or_colic', 
                    'complaint_contains_dental', 'complaint_contains_device_complication', 
                    'complaint_contains_diarrhea', 'complaint_contains_ear_complaint', 
                    'complaint_contains_epistaxis', 'complaint_contains_extremity', 
                    'complaint_contains_eye_complaint', 'complaint_contains_syncope', 
                    'complaint_contains_foreign_body', 'complaint_contains_fever', 
                    'complaint_contains_follow_up', 'complaint_contains_general', 
                    'complaint_contains_gi_bleed', 'complaint_contains_gynecologic', 
                    'complaint_contains_head_or_neck', 'complaint_contains_headache', 
                    'complaint_contains_laceration', 'complaint_contains_lump_or_mass', 
                    'complaint_contains_male_genital', 'complaint_contains_mvc', 
                    'complaint_contains_neck_pain', 'complaint_contains_neurologic', 
                    'complaint_contains_poisoning', 'complaint_contains_poor_feeding', 
                    'complaint_contains_primary_care', 'complaint_contains_psych', 'complaint_contains_rash', 
                    'complaint_contains_other_respiratory', 'complaint_contains_seizure', 
                    'complaint_contains_sore_throat', 'complaint_contains_trauma', 'complaint_contains_urinary', 
                    'complaint_contains_vomiting']
data_bin.loc[:, complaint_groups] = data[complaint_groups].fillna(0)

/var/folders/kd/dr1q38qs78vg1_j0g3xkr8vw0000gn/T/ipykernel_60314/4271351638.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_bin[complaint_groups] = data[complaint_groups].fillna(0)
/var/folders/kd/dr1q38qs78vg1_j0g3xkr8vw0000gn/T/ipykernel_60314/4271351638.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_bin[complaint_groups] = data[complaint_groups].fillna(0)
/var/folders/kd/dr1q38qs78vg1_j0g3xkr8vw0000gn/T/ipykernel_60314/4271351638.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

In [23]:
data_bin.to_csv('preprocessed_CHLA.csv')

In [24]:
# Columns in binarized data
print(data_bin.columns.tolist())

['id_visit', 'id_patient', 'age', 'is_eighteen_months_to_3_years', 'is_five_to_10_years', 'is_six_to_12_months', 'is_ten_to_15_years', 'is_three_to_5_years', 'is_three_to_6_months', 'is_twelve_to_18_months', 'is_under_3_months', 'is_unknown', 'is_male', 'is_asian', 'is_hispanic', 'is_hispanic_white', 'is_non_hispanic_black', 'is_other', 'arrival_EMS', 'arrival_unknown', 'arrival_other', 'is_language_spanish', 'is_language_mandarin', 'is_language_russian', 'is_language_armenian', 'is_language_other', 'state_out_of_state', 'state_unknown', 'miles_travelled', 'SDI_score', 'insurance_private', 'insurance_unknown', 'is_admitted', 'is_transfer', 'num_previous_admissions', 'num_previous_visits_without_admission', 'num_patients_at_arrival', 'icd_score', 'weight', 'norm_heart_rate', 'norm_respiratory_rate', 'syst_diff_pals', 'temp_fever', 'arrival_year_2019', 'arrival_year_2020', 'arrival_year_2021', 'arrival_year_2022', 'arrival_year_2023', 'arrival_year_2024', 'arrival_year_2025', 'arrival_se